# 05 — Evaluation & Hybrid Comparison

Brings everything together on the held-out **test set**:

* Pure MLP performance (per-class metrics, confusion matrix)
* Pure Autoencoder performance (binary attack / benign)
* **Hybrid** = AE flags anomaly, MLP classifies — same logic the production
  `SmartTIDS_Predictor` runs.
* SHAP explanations on a sample
* Inference latency


In [ ]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

from src import config as cfg
from src.evaluation import (
    plot_confusion, plot_reconstruction_error, autoencoder_anomaly_metrics,
)
from src.inference import SmartTIDS_Predictor
sns.set_theme(style="whitegrid")


## 1. Load test set and artifacts

In [ ]:
test = pd.read_csv(cfg.PROCESSED_DIR / "test.csv")
scaler = joblib.load(cfg.SCALER_FILE)
with open(cfg.FEATURE_NAMES_FILE) as f: feature_names = json.load(f)
with open(cfg.LABEL_MAP_FILE)    as f: label_map = {int(k):v for k,v in json.load(f).items()}
inv_map = {v:k for k,v in label_map.items()}

test = test[test[cfg.LABEL_COL].isin(inv_map)].reset_index(drop=True)
y_true = test[cfg.LABEL_COL].map(inv_map).astype(int).values
X = scaler.transform(test[feature_names].astype(np.float32).values)

mlp = tf.keras.models.load_model(cfg.MLP_MODEL_FILE, compile=False)
ae  = tf.keras.models.load_model(cfg.AE_MODEL_FILE,  compile=False)
ae_threshold = json.loads(cfg.AE_THRESHOLD_FILE.read_text())["threshold"]
class_names = [label_map[i] for i in range(len(label_map))]
print("Test rows:", len(test))


## 2. MLP-only metrics

In [ ]:
y_pred_mlp = mlp.predict(X, batch_size=4096, verbose=0).argmax(axis=1)
print(classification_report(y_true, y_pred_mlp, target_names=class_names,
                            digits=4, zero_division=0))
plot_confusion(y_true, y_pred_mlp, class_names, normalize=True,
               title="MLP — normalised confusion matrix")
plt.show()


## 3. Autoencoder-only metrics (binary)

In [ ]:
recon = ae.predict(X, batch_size=4096, verbose=0)
err = np.mean((X - recon) ** 2, axis=1)
y_is_attack = (y_true != inv_map["BENIGN"]).astype(int)

err_benign = err[y_is_attack == 0]
err_attack = err[y_is_attack == 1]
plot_reconstruction_error(err_benign, err_attack, ae_threshold,
                          title="Autoencoder MSE on test set")
plt.show()

ae_metrics = autoencoder_anomaly_metrics(err, y_is_attack, ae_threshold)
pd.DataFrame([ae_metrics])


## 4. Hybrid — AE gates / MLP classifies

Decision rule (matches `SmartTIDS_Predictor`):

* `is_anomaly = err > threshold`
* if MLP says BENIGN but AE flags anomaly  -> mark `UNKNOWN_ANOMALY`
* otherwise                                -> MLP top class


In [ ]:
mlp_proba = mlp.predict(X, batch_size=4096, verbose=0)
mlp_top   = mlp_proba.argmax(axis=1)
mlp_conf  = mlp_proba.max(axis=1)
anomaly   = err > ae_threshold

UNKNOWN_ID = -1
hybrid_pred = np.where(
    (mlp_top == inv_map["BENIGN"]) & anomaly,
    UNKNOWN_ID,
    mlp_top,
)

# Score against multi-class ground truth, ignoring UNKNOWN bucket.
mask = hybrid_pred != UNKNOWN_ID
print(f"UNKNOWN flagged: {(~mask).sum()} / {len(mask)} rows")
print(classification_report(
    y_true[mask], hybrid_pred[mask],
    target_names=class_names, digits=4, zero_division=0,
))


## 5. SHAP explanations (sample of 500 rows)

In [ ]:
import shap

sample_idx = np.random.RandomState(0).choice(len(X), size=500, replace=False)
explainer = shap.DeepExplainer(mlp, X[np.random.RandomState(1).choice(len(X), 200)])
shap_values = explainer.shap_values(X[sample_idx])

# Aggregate |SHAP| across classes -> overall feature importance
agg = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
imp = pd.DataFrame({"feature": feature_names, "importance": agg}) \
        .sort_values("importance", ascending=False).head(20)

plt.figure(figsize=(8, 6))
sns.barplot(data=imp, x="importance", y="feature", color="steelblue")
plt.title("Top-20 features (mean |SHAP| across classes)")
plt.tight_layout(); plt.show()


## 6. Inference latency — production predictor

In [ ]:
predictor = SmartTIDS_Predictor()

# warm-up
_ = predictor.predict_batch(test.head(32))

N = 1000
sample_df = test.head(N)
t0 = time.perf_counter()
results = predictor.predict_batch(sample_df)
elapsed = time.perf_counter() - t0
print(f"Batch of {N}: {elapsed*1000:.1f} ms total -> "
      f"{elapsed/N*1000:.3f} ms/flow ({N/elapsed:,.0f} flows/sec)")

print("\nExample result:")
print(json.dumps(results[0], indent=2))


## 7. Healthcheck

In [ ]:
print(json.dumps(predictor.healthcheck(), indent=2))
